# Get WhisperTimeSync

In [1]:
# !rm -rf WhisperTimeSync
# !git clone https://github.com/EtienneAb3d/WhisperTimeSync.git

In [2]:
# !apt install -y ffmpeg
# !pip install scipy soundfile tqdm six torch transformers vad librosa srt numpy==2.0

# Transcribe

In [3]:
import json
from get_torah_text_using_sefaria import get_chapter_string
import scipy.io.wavfile as wavfile
import io
from six.moves.urllib.request import urlopen
import pathlib
import soundfile as sf
from tqdm import tqdm
from nikud_and_teamim import remove_nikud, replace_teamim_with_emphasis, remove_nikud_and_teamim
from remove_nikud_dicta import remove_nikud_dicta

import torch
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import librosa
import srt
from datetime import timedelta
from transformers import pipeline
import os
import soundfile as sf
import scipy.signal
from vad import EnergyVAD 
from tqdm import tqdm




!nvidia-smi


/home/prj8045/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Mon Dec  9 15:05:51 2024       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.107.02             Driver Version: 550.107.02     CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A5000               Off |   00000000:01:00.0 Off |                  Off |
| 30%   24C    P8             17W /  230W |   14417MiB /  24564MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# choose the GPU with empty memory
mem = !nvidia-smi --query-gpu=memory.free --format=csv
mem = mem[1:]
mem = [int(m.replace(' MiB', '')) for m in mem]
device = torch.device(f'cuda:{mem.index(max(mem))}' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda', index=1)

we want to adapt the text to time, in iterations.
1) Ketiv Maleh (בֹּקֶר -> בוקר) - becuse we use hebrew model that trained on modern hebrew in without nikud in Ketiv Male this is the first step to adapt the text to the model.
2) Ketiv Haser without teamim - it closer to our data.
3) Ketiv Haser with teamim - now it is much simpler to adapt the text to the model.

In [6]:
# Load the fine-tuned model and processor
# model = WhisperForConditionalGeneration.from_pretrained("ivrit-ai/whisper-v2-pd1-e1").to(device)
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-large-v2").to(device)
processor = WhisperProcessor.from_pretrained("openai/whisper-large-v2")

model.generation_config.language = "he"

# Define maximum audio segment length
MAX_SEGMENT_LENGTH = 30 * 1000  # 30 seconds in milliseconds



In [7]:
WITH_TIMESTAMPS = False # True, False, or "word". True will use the first method, "word" is the best method for timestamps in whisper
MAX_SEGMENT_LENGTH_CHARS = 22 # maximum number of characters in each segment in the SRT file 

In [8]:
from audio_to_srt import generate_srt_from_audio
from create_steps_files import create_steps_files, remove_steps_files

In [11]:
whisperTimeSync = "/home/prj8045/Torah-reading-data--alignment-and-slicer/automatic using WhisperTimeSync/WhisperTimeSync/distrib/WhisperTimeSync.jar"
def iterative_sync_text_srt(text_path, whisperTimeSync):
    """
    Iteratively synchronize the text and the low-quality SRT file.
    """
    # Load the text and the low-quality SRT file

    create_steps_files(text_path)
    !java -Xmx2G -jar "{whisperTimeSync}" "output.srt" "step01.txt" he
    !java -Xmx2G -jar "{whisperTimeSync}" "step01.txt.srt" "step02.txt" he
    !java -Xmx2G -jar "{whisperTimeSync}" "step02.txt.srt" "step03.txt" he
    !java -Xmx2G -jar "{whisperTimeSync}" "step03.txt.srt" "step04.txt" he
    !java -Xmx2G -jar "{whisperTimeSync}" "step04.txt.srt" "final_step.txt" he
    

In [ ]:
with open("/home/prj8045/data/data.json", "r") as f:
    data = json.load(f)[:3]
for nusach in [("ashkenazi","ak"), ("sefaradi","sk")]: # the first is the name of the nusach, the second is the prefix of the audio files
    for datum in data:
        for i in range(1, 8):
            audio_path = f"/home/prj8045/data/{nusach[0]}/{nusach[1]}{datum['parent']}-{datum['value']}{i}.mp3"
            generate_srt_from_audio(audio_path, "output.srt", batch_size=32)
            
            # os.rename("output.srt", f"srts/{audio_path.split('/')[-1].replace('.mp3', '.srt')}")
            
            
            # Sync high quality text using our iterative method
            text_path = f"/home/prj8045/Torah-reading-data--alignment-and-slicer/text/{datum['name']}-{i}.txt"
            iterative_sync_text_srt(text_path, whisperTimeSync)

            # # rename the final_step.txt.srt to the name of the audio file, and move it to the the same folder as the audio file
            os.rename("final_step.txt.srt", f"{audio_path.replace('.mp3', '.srt')}")
remove_steps_files()

  0%|          | 0/1 [00:00<?, ?it/s]/home/prj8045/.local/lib/python3.10/site-packages/transformers/models/whisper/generation_whisper.py:509: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
100%|██████████| 1/1 [00:08<00:00,  8.40s/it]



Output (step01.txt.srt):

Output (step02.txt.srt):

Output (step03.txt.srt):

Output (step04.txt.srt):

Output (final_step.txt.srt):


100%|██████████| 1/1 [00:04<00:00,  4.84s/it]



Output (step01.txt.srt):

Output (step02.txt.srt):

Output (step03.txt.srt):

Output (step04.txt.srt):

Output (final_step.txt.srt):


100%|██████████| 1/1 [00:08<00:00,  8.02s/it]


...


Output (step01.txt.srt):

Output (step02.txt.srt):

Output (step03.txt.srt):
^C

Output (step04.txt.srt):

Output (final_step.txt.srt):


100%|██████████| 1/1 [00:04<00:00,  4.85s/it]



Output (step01.txt.srt):

Output (step02.txt.srt):

Output (step03.txt.srt):

Output (step04.txt.srt):

Output (final_step.txt.srt):


100%|██████████| 1/1 [00:04<00:00,  4.39s/it]



Output (step01.txt.srt):

Output (step02.txt.srt):

Output (step03.txt.srt):

Output (step04.txt.srt):


In [9]:
dir = "/home/prj8045/Torah-reading-data--alignment-and-slicer/automatic using WhisperTimeSync/"
# Call the main function
generate_srt_from_audio(dir + "/audio.mp3", dir + "output.srt", batch_size=16)

  0%|          | 0/2 [00:00<?, ?it/s]/home/prj8045/.local/lib/python3.10/site-packages/transformers/models/whisper/generation_whisper.py:509: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, None], [2, 50359]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.
100%|██████████| 2/2 [00:12<00:00,  6.41s/it]


In [10]:
dir = "/home/prj8045/Torah-reading-data--alignment-and-slicer/automatic using WhisperTimeSync/"
# Call the main function
generate_srt_from_audio(dir + "audio.mp3", dir + "output.srt", batch_size=16)

  0%|          | 0/22 [00:00<?, ?it/s]/home/prj8045/.local/lib/python3.10/site-packages/transformers/models/whisper/generation_whisper.py:509: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
You have passed task=transcribe, but also have set `forced_decoder_ids` to [[1, None], [2, 50359]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of task=transcribe.
  9%|▉         | 2/22 [00:05<00:54,  2.74s/it]


KeyboardInterrupt: 

In [11]:
adsfs

NameError: name 'adsfs' is not defined

In [12]:
!apt install -y java

E: Could not open lock file /var/lib/dpkg/lock-frontend - open (13: Permission denied)
E: Unable to acquire the dpkg frontend lock (/var/lib/dpkg/lock-frontend), are you root?


In [12]:
# # Now we have the srt file, with time but the low quality text, and the text file with the high quality text.
# # We will use the WhisperTimeSync to sync the two files, and get the srt file with the high quality text.
# !java -Xmx2G -jar WhisperTimeSync/distrib/WhisperTimeSync.jar "output.srt" "text.txt" he


In [1]:
# We will run the WhisperTimeSync on the output and the step files
!java -Xmx2G -jar WhisperTimeSync/distrib/WhisperTimeSync.jar "output.srt" "step01.txt" he
!java -Xmx2G -jar WhisperTimeSync/distrib/WhisperTimeSync.jar "step01.txt.srt" "step02.txt" he
!java -Xmx2G -jar WhisperTimeSync/distrib/WhisperTimeSync.jar "step02.txt.srt" "step03.txt" he
!java -Xmx2G -jar WhisperTimeSync/distrib/WhisperTimeSync.jar "step03.txt.srt" "step04.txt" he
!java -Xmx2G -jar WhisperTimeSync/distrib/WhisperTimeSync.jar "step04.txt.srt" "final_step.txt" he


/bin/bash: line 1: java: command not found


/bin/bash: line 1: java: command not found
/bin/bash: line 1: java: command not found
/bin/bash: line 1: java: command not found
/bin/bash: line 1: java: command not found


In [1]:
# # try do it in one step
!java -Xmx2G -jar WhisperTimeSync/distrib/WhisperTimeSync.jar "output.srt" "final_step.txt" he

/bin/bash: line 1: java: command not found


# The method for Milra and Milael future project:

In [15]:
# WITH_TEAMIM = False
# WITH_STRESS = True
# SR = 16000


# if WITH_TEAMIM and WITH_STRESS:
#     print("WITH_TEAMIM and WITH_STRESS cannot be True at the same time.")
#     exit(1)


In [16]:
# Example with the first chapter of Genesis:
with open('links_for_audio_from_929.json', 'r') as f:
    links_for_audio_from_929 = json.load(f)

url = links_for_audio_from_929["books"][0]["chapters"][0]["link"]

# Download the audio file
z = io.BytesIO(urlopen(url).read())
pathlib.Path((f"{book_name}_{chapter}.mp3")).write_bytes(z.getbuffer())
audio, sr = librosa.load(f"{book_name}_{chapter}.mp3", sr=SR)


# Get the text from Sefaria
book_name = "Genesis"
chapter = 1
text = get_chapter_string(book_name, chapter)

# remove the nikud and replace the teamim with empsis
if WITH_TEAMIM:
    text = remove_nikud(text)
else:
    if WITH_STRESS:
        text = remove_nikud(text)
        text = replace_teamim_with_emphasis(text)
    else:
        text = remove_nikud_and_teamim(text)
text = "בראשֽית " + "פרק " + "אלף " + text

with open(f"{book_name}_{chapter}.txt", "w") as f:
    f.write(text)

# Transcribe the audio to get the timestamps (as srt file)
!python3 WhisperTimeSync/transcribe.py /content/"{book_name}_{chapter}.mp3" large-v3

# Align(sync) the text with the audio
!java -Xmx2G -jar WhisperTimeSync/distrib/WhisperTimeSync.jar "{book_name}_{chapter}.srt" "{book_name}_{chapter}.txt" he

# Load the srt file
with open(f"{book_name}_{chapter}.txt.srt", "r") as f:
    srt = f.read()

dataset = {"text": [], "start": [], "end": [], "audio_file": []}
# Add the times, text, and audio file name(save each sentence as a separate audio file), to the dataset
for i, line in enumerate(srt.split("\n")):
    if i % 4 == 0:
        dataset["start"].append(line)
    elif i % 4 == 1:
        dataset["end"].append(line)
    elif i % 4 == 2:
        dataset["text"].append(line)
    elif i % 4 == 3:
        dataset["audio_file"].append(f"{book_name}_{chapter}_{i//4}.wav")
        start = int(float(dataset["start"][-1].replace(",", ".").replace(" --> ", "")) * SR)
        end = int(float(dataset["end"][-1].replace(",", ".").replace(" --> ", "")) * SR)
        sf.write(f"{book_name}_{chapter}_{i//4}.wav", audio[start:end], SR)


FileNotFoundError: [Errno 2] No such file or directory: 'links_for_audio_from_929.json'

In [ ]:
import librosa
import requests
import json
from get_torah_text_using_sefaria import get_chapter_string
import scipy.io.wavfile as wavfile
import io
from six.moves.urllib.request import urlopen
import pathlib
import soundfile as sf
from tqdm import tqdm
SR = 16000


books_data = [
    {"name": "Bereshit", "number": 1, "chapters": 50, "sefaria_name": "Genesis"},
    {"name": "Shemot", "number": 2, "chapters": 40, "sefaria_name": "Exodus"},
    {"name": "Vaikra", "number": 3, "chapters": 27, "sefaria_name": "Leviticus"},
    {"name": "Bamidbar", "number": 4, "chapters": 36, "sefaria_name": "Numbers"},
    {"name": "Dvarim", "number": 5, "chapters": 34, "sefaria_name": "Deuteronomy"}
]
sefer_names = ["", "בְּרֵאשִֽׁית", "שְׁמֽות", "וַיִּקְרָֽא", "בַּמִּדְבָּֽר", "דְּבָרִֽים"]
otiyot_gematria = ["", "אָֽלֶף", "בֵּֽית", "גִּֽימֶל", "דָּֽלֶת", "הֵֽא", "וָֽו", "זַֽיִן", "חֵֽית", "טֵֽית", "יֽוֹד", "יֽוֹד-אָֽלֶף", "יֽוֹד-בֵּֽית", "יֽוֹד-גִּֽימֶל", "יֽוֹד-דָּֽלֶת", "טֵֽית-וָֽו", "טֵֽית-זַֽיִן", "יֽ-זַֽיִן", "יֽוֹד-חֵֽית", "יֽוֹד-טֵֽית", "כַּֽף", "כַּֽף-אָֽלֶף", "כַּֽף-בֵּֽית", "כַּֽף-גִּֽימֶל", "כַּֽף-דָּֽלֶת", "כַּֽף-הֵֽא", "כַּֽף-וָֽו", "כַּֽף-זַֽיִן", "כַּֽף-חֵֽית", "כַּֽף-טֵֽית", "לָֽמֶד", "לָֽמֶד-אָֽלֶף", "לָֽמֶד-בֵּֽית", "לָֽמֶד-גִּֽימֶל", "לָֽמֶד-דָּֽלֶת", "לָֽמֶד-הֵֽא", "לָֽמֶד-וָֽו", "לָֽמֶד-זַֽיִן", "לָֽמֶד-חֵֽית", "לָֽמֶד-טֵֽית", "מֵֽם", "מֵֽם-אָֽלֶף", "מֵֽם-בֵּֽית", "מֵֽם-גִּֽימֶל", "מֵֽם-דָּֽלֶת", "מֵֽם-הֵֽא", "מֵֽם-וָֽו", "מֵֽם-זַֽיִן", "מֵֽם-חֵֽית", "מֵֽם-טֵֽית", "נֽוּן", "נֽוּן-אָֽלֶף", "נֽוּן-בֵּֽית", "נֽוּן-גִּֽימֶל", "נֽוּן-דָּֽלֶת", "נֽוּן-הֵֽא", "נֽוּן-וָֽו", "נֽוּן-זַֽיִן", "נֽוּן-חֵֽית", "נֽוּן-טֵֽית", "סָֽמֶךְ"]
otiyot_gematria = [remove_nikud(ot) for ot in otiyot_gematria]
dataset = {"text": [], "audio_file": []}

with open('links_for_audio_from_929.json', 'r') as f:
    links_for_audio_from_929 = json.load(f)

# Calculate total number of chapters
total_chapters = sum(book["chapters"] for book in books_data)

# Create a progress bar
pbar = tqdm(total=total_chapters)

# Loop over all the chapters of all the books
for book in books_data:
    book_name = book["name"]
    sefaria_name = book["sefaria_name"]
    for chapter in range(1, book["chapters"]+1):
        if chapter % 27 != 0: # test on small data
            continue
        url = links_for_audio_from_929["books"][book["number"]-1]["chapters"][chapter-1]["link"]
        z = io.BytesIO(urlopen(url).read())
        pathlib.Path((f"{book_name}_{chapter}.mp3")).write_bytes(z.getbuffer())
        audio, sr = librosa.load(f"{book_name}_{chapter}.mp3", sr=SR)
        text = get_chapter_string(sefaria_name, chapter)
        if WITH_TEAMIM:
            text = remove_nikud(text)
        else:
            if WITH_STRESS:
                text = remove_nikud(text)
                text = replace_teamim_with_emphasis(text)
            else:
                text = remove_nikud_and_teamim(text)
        text = f"{sefer_names[book['number']]} " + f"פרק {otiyot_gematria[chapter]} " + text
        with open(f"{book_name}_{chapter}.txt", "w") as f:
            f.write(text)
        !python3 WhisperTimeSync/transcribe.py /content/"{book_name}_{chapter}.mp3" large-v3
        !java -Xmx2G -jar WhisperTimeSync/distrib/WhisperTimeSync.jar "{book_name}_{chapter}.srt" "{book_name}_{chapter}.txt" he
        with open(f"{book_name}_{chapter}.txt.srt", "r") as f:
            srt = f.read()

        for i, line in enumerate(srt.split("\n")):
            if i % 4 == 0: # The line is a number of the subtitle
                continue
            if i % 4 == 1: # The line is a time
                start, end = line.split(" --> ")
                start = start.replace(",", ".")
                end = end.replace(",", ".")
                start = start.split(":")
                end = end.split(":")
                start = float(start[0])*3600 + float(start[1])*60 + float(start[2])
                end = float(end[0])*3600 + float(end[1])*60 + float(end[2])
                start = float(start)
                end = float(end)
                continue
            if i % 4 == 2: # The line is the text
                dataset["text"].append(line)
                dataset["audio_file"].append(f"{book_name}_{chapter}_{int(i/4):02}.wav")
                # Save the audio file
                sf.write(f"{book_name}_{chapter}_{int(i/4):02}.wav", audio[int(start*SR):int(end*SR)], SR)
            if i % 4 == 3: # The line is an empty line
                continue
        # Remove the original audio file and the srt file
        !rm "{book_name}_{chapter}.mp3"
        !rm "{book_name}_{chapter}.txt"
        !rm "{book_name}_{chapter}.txt.srt"
        !rm "{book_name}_{chapter}.srt"

        # Update the progress bar
        pbar.update()

# Save the dataset
with open("dataset.json", "w") as f:
    json.dump(dataset, f)
print("Done")

In [ ]:

book_name = "Bereshit"
chapter = 27
url = links_for_audio_from_929["books"][1]["chapters"][chapter-1]["link"]
z = io.BytesIO(urlopen(url).read())
pathlib.Path((f"{book_name}_{chapter}.mp3")).write_bytes(z.getbuffer())
!python3 WhisperTimeSync/transcribe.py /content/"{book_name}_{chapter}.mp3" large-v3

In [ ]:
# Save the dataset
with open("dataset.json", "w", encoding="utf-8") as f:
    json.dump(dataset, f)

In [ ]:
# play random audio and print its text
import random
import IPython.display as ipd
index = random.randint(0, len(dataset["audio_file"]))
print(dataset["text"][index])
ipd.Audio(dataset["audio_file"][index])


In [ ]:
index = 14
print(dataset["text"][index])
ipd.Audio(dataset["audio_file"][index])

In [ ]:
# Sync the times of the real text using the srt file
!java -Xmx2G -jar WhisperTimeSync/distrib/WhisperTimeSync.jar "{book_name}_{chapter}.srt" "{book_name}_{chapter}.txt" he

In [ ]:
books_data = [
    {"name": "Bereshit", "number": 1, "chapters": 50, "sefaria_name": "Genesis"},
    {"name": "Shemot", "number": 2, "chapters": 40, "sefaria_name": "Exodus"},
    {"name": "Vaikra", "number": 3, "chapters": 27, "sefaria_name": "Leviticus"},
    {"name": "Bamidbar", "number": 4, "chapters": 36, "sefaria_name": "Numbers"},
    {"name": "Dvarim", "number": 5, "chapters": 34, "sefaria_name": "Deuteronomy"}
]
sefer_names = ["", "בְּרֵאשִֽׁית", "שְׁמֽות", "וַיִּקְרָֽא", "בַּמִּדְבָּֽר", "דְּבָרִֽים"]
otiyot_gematria = ["", "אָֽלֶף", "בֵּֽית", "גִּֽימֶל", "דָּֽלֶת", "הֵֽא", "וָֽו", "זַֽיִן", "חֵֽית", "טֵֽית", "יֽוֹד", "יֽוֹד-אָֽלֶף", "יֽוֹד-בֵּֽית", "יֽוֹד-גִּֽימֶל", "יֽוֹד-דָּֽלֶת", "טֵֽית-וָֽו", "טֵֽית-זַֽיִן", "יֽ-זַֽיִן", "יֽוֹד-חֵֽית", "יֽוֹד-טֵֽית", "כַּֽף", "כַּֽף-אָֽלֶף", "כַּֽף-בֵּֽית", "כַּֽף-גִּֽימֶל", "כַּֽף-דָּֽלֶת", "כַּֽף-הֵֽא", "כַּֽף-וָֽו", "כַּֽף-זַֽיִן", "כַּֽף-חֵֽית", "כַּֽף-טֵֽית", "לָֽמֶד", "לָֽמֶד-אָֽלֶף", "לָֽמֶד-בֵּֽית", "לָֽמֶד-גִּֽימֶל", "לָֽמֶד-דָּֽלֶת", "לָֽמֶד-הֵֽא", "לָֽמֶד-וָֽו", "לָֽמֶד-זַֽיִן", "לָֽמֶד-חֵֽית", "לָֽמֶד-טֵֽית", "מֵֽם", "מֵֽם-אָֽלֶף", "מֵֽם-בֵּֽית", "מֵֽם-גִּֽימֶל", "מֵֽם-דָּֽלֶת", "מֵֽם-הֵֽא", "מֵֽם-וָֽו", "מֵֽם-זַֽיִן", "מֵֽם-חֵֽית", "מֵֽם-טֵֽית", "נֽוּן", "נֽוּן-אָֽלֶף", "נֽוּן-בֵּֽית", "נֽוּן-גִּֽימֶל", "נֽוּן-דָּֽלֶת", "נֽוּן-הֵֽא", "נֽוּן-וָֽו", "נֽוּן-זַֽיִן", "נֽוּן-חֵֽית", "נֽוּן-טֵֽית", "סָֽמֶךְ"]
otiyot_gematria = [remove_nikud(ot) for ot in otiyot_gematria]


# Load the data of links_for_audio_from_929.json

with open('WhisperTimeSync/links_for_audio_from_929.json', 'r') as f:
    links_for_audio_from_929 = json.load(f)


import requests
url = links_for_audio_from_929["books"][0]["chapters"][0]["link"]
# Download (to RAM) the audio file for the first chapter of Bereshit
r = requests.get(url, allow_redirects=True)


# for book in books_data:
#     print(book['name'])
#     for chapter in range(1, book['chapters'] + 1):
#         print(chapter)
#         print(links_for_audio_from_929[book['sefaria_name']][str(chapter)])


#         להוסיף את ההורדה של האודיו ומשיכת הטקסט הרלוונטי
#         ואז להפעיל את הקוד של וויספרטיימסינק על הטקסט והאודיו







In [ ]:
!cat /content/51.Shmot_1.srt

# Synchronize

In [ ]:
!java -Xmx2G -jar WhisperTimeSync/distrib/WhisperTimeSync.jar /content/51.Shmot_1.srt /content/Exodus.1.txt he

In [ ]:
!cat /content/Exodus.1.txt.srt